# MCQ Distractor Generation (DG) -- LoRA Training
**Target:** T4 (15 GB)  |  **Model:** `unsloth/Qwen3-4B-Instruct`

**Setup:** Upload `dg_formatted.jsonl` as a Kaggle Dataset, set `DATA_PATH` in Cell 6 if needed, run all.


## 1. GPU Detection


In [ ]:
import os
# ── Pin to single GPU 0 — Unsloth does not support multi-GPU DDP ──────────
# Kaggle exposes 2x T4 but Unsloth can only use one.  Setting this env var
# before any CUDA call ensures the process never sees the second card,
# preventing any accidental DataParallel / device-mismatch errors.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # silence HF tokenizer warning
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Enable GPU T4 x2 in Notebook Settings.")

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
n = torch.cuda.device_count()
print(f"GPU: {name}  |  VRAM: {vram:.1f} GB  |  Count: {n}  |  CUDA: {torch.version.cuda}")

if "T4" not in name:
    raise RuntimeError(
        f"This notebook requires GPU T4 x2. Got: {name}.\n"
        f"Go to Settings → Accelerator → GPU T4 x2"
    )

GPU_TYPE = "T4"
print(f"✅ GPU T4 detected. Proceeding.")


## 2. Install Dependencies


In [ ]:
import subprocess
def _install(cmd, label):
    print(f"[INSTALL] {label} ...", end=" ")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print("OK" if r.returncode == 0 else f"FAIL: {r.stderr[-200:]}")

# Force PyTorch with CUDA 12.1 kernels (supports P100 sm_60 + T4 sm_75)
_install("pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q", "PyTorch cu121")
_install("pip install unsloth --upgrade -q", "unsloth")
_install("pip install --upgrade trl transformers peft datasets -q", "trl+transformers+peft+datasets")
_install("pip install rouge-score sentence-transformers scikit-learn structlog numpy -q", "eval deps")

from unsloth import FastLanguageModel
print("\nUnsloth import OK")


## 3. Configuration


In [ ]:
import math
from pathlib import Path

DATA_PATH  = "/kaggle/input/mcq-training-data/dg_formatted.jsonl"
OUTPUT_DIR = "/kaggle/working/mcq_dg"
BASE_MODEL = "unsloth/Qwen3-4B-Instruct"

# Single T4 (15 GB VRAM) -- Unsloth 4-bit.
# Effective batch = 4 x 4 = 16.  DG inputs are short (~300 tokens);
# packing fills 1024-token windows very efficiently.
BATCH_SIZE     = 4
GRAD_ACCUM     = 4
MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT   = True

# DG LoRA: simpler task than QG, lower rank is sufficient.
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05

EPOCHS = 2
LR     = 2e-4
WD     = 0.01

# T4 does not support bf16; use fp16.
USE_FP16 = True
USE_BF16 = False

# Warmup = 5% of total training steps (computed after data load).
WARMUP_RATIO = 0.05

print(f"Config: batch={BATCH_SIZE} | accum={GRAD_ACCUM} | eff_batch={BATCH_SIZE*GRAD_ACCUM}")
print(f"seq_len={MAX_SEQ_LENGTH} | 4bit={LOAD_IN_4BIT} | r={LORA_R} | alpha={LORA_ALPHA}")
print(f"Model: {BASE_MODEL}")

## 4. Prompt & Parser Helpers


In [ ]:
import re

# Simplified DG training system prompt (~100 tokens).
# Matches what format_dg.py wrote into the 'text' field.
# Used during eval to reconstruct the inference prompt from val sample fields.
_DG_SYSTEM = (
    "You are an expert distractor generator for CS multiple-choice questions.\n\n"
    "Generate ONE plausible but incorrect answer that a student with a\n"
    "specific misconception might choose.\n\n"
    "TWO SIGNALS CONTROL THE DISTRACTOR:\n"
    "  mastery_level controls distractor sophistication:\n"
    "    Novice: clearly wrong to anyone who read carefully\n"
    "    Intermediate: requires careful reasoning to eliminate\n"
    "    Expert: plausible to someone with partial understanding,\n"
    "            technically adjacent but ultimately wrong\n\n"
    "  score_category controls distractor difficulty:\n"
    "    very_weak: easy to eliminate -- student needs confidence\n"
    "    weak: plausible but distinguishable with effort\n"
    "    moderate: standard plausibility\n"
    "    strong: as subtle and tricky as mastery allows\n\n"
    "OUTPUT FORMAT (required):\n"
    "DISTRACTOR: <the incorrect answer>\n\n"
    "The distractor must be factually wrong, distinct from the correct\n"
    "answer, and match the sophistication and difficulty specified above."
)


def build_dg_eval_prompt(question, correct_answer, question_type,
                          mastery_level, score_category,
                          distractor_category="general", chunk_text=""):
    # Reconstruct the compact user turn used during training, for eval inference.
    user = (
        f"generate distractor: type: {question_type} | mastery: {mastery_level} | "
        f"score_category: {score_category} | category: {distractor_category} | "
        f"question: {question} | answer: {correct_answer} | content: {chunk_text}"
    )
    return [{"role": "system", "content": _DG_SYSTEM},
            {"role": "user",   "content": user}]


def apply_chat(messages, tokenizer):
    # Apply chat template with Qwen3 non-thinking mode.
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)


def extract_dg_output(raw):
    if not raw or not raw.strip():
        return None
    t = raw.strip()
    m = re.search(r"DISTRACTOR:\s*(.+)", t, re.DOTALL)
    if m:
        d = m.group(1).strip().split("\n")[0].strip()
        if d:
            return d
    lines = [l.strip() for l in t.split("\n") if l.strip()]
    if len(lines) == 1 and not lines[0].startswith(("QUESTION:", "ANSWER:", "EXPLANATION:")):
        return lines[0]
    return None


print("Helpers loaded OK")

## 5. Load & Split Data


In [ ]:
import json
import random
from collections import defaultdict
from sklearn.model_selection import train_test_split

def load_jsonl(path):
    samples = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                try: samples.append(json.loads(line))
                except json.JSONDecodeError: pass
    return samples

def stratified_split_dg(samples, val_ratio=0.1):
    # Shuffle before splitting so sequential book order in the source
    # file does not bias the train/val split or batch composition.
    random.seed(42)
    random.shuffle(samples)
    if len(samples) < 10:
        i = max(1, int(len(samples)*(1-val_ratio))); return samples[:i], samples[i:]
    labels = [f"{s.get('question_type','?')}_{s.get('mastery_level','?')}" for s in samples]
    lc = defaultdict(int)
    for lb in labels: lc[lb] += 1
    labels_clean = [lb if lc[lb]>=2 else "rare" for lb in labels]
    if len(set(labels_clean)) < 2:
        i = max(1, int(len(samples)*(1-val_ratio))); return samples[:i], samples[i:]
    return train_test_split(samples, test_size=val_ratio, stratify=labels_clean, random_state=42)

all_samples = load_jsonl(DATA_PATH)
print(f"Loaded {len(all_samples)} DG samples")
train_samples, val_samples = stratified_split_dg(all_samples)
print(f"Train: {len(train_samples)} | Val: {len(val_samples)}")


## 6. Load Model + Apply LoRA


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=LOAD_IN_4BIT, dtype=None,
    device_map={"": 0})

model = FastLanguageModel.get_peft_model(model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", use_gradient_checkpointing="unsloth", random_state=42)

tp = sum(p.numel() for p in model.parameters() if p.requires_grad)
tt = sum(p.numel() for p in model.parameters())
print(f"Trainable: {tp:,} / {tt:,} ({100*tp/tt:.3f}%)")


## 7. Build Dataset


In [ ]:
from datasets import Dataset
train_dataset = Dataset.from_list(train_samples)
val_dataset = Dataset.from_list(val_samples)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")


## 8. Train


In [ ]:
import time
from pathlib import Path
from trl import SFTTrainer, SFTConfig

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Compute warmup steps = 5% of total training steps.
_steps_per_epoch = max(1, len(train_samples) // (BATCH_SIZE * GRAD_ACCUM))
_total_steps     = _steps_per_epoch * EPOCHS
WARMUP           = max(1, round(WARMUP_RATIO * _total_steps))
print(f"steps/epoch~{_steps_per_epoch} | total_steps={_total_steps} | warmup={WARMUP}")

CKPT_DIR = f"{OUTPUT_DIR}/checkpoints"
args = SFTConfig(
    output_dir=CKPT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, warmup_steps=WARMUP, weight_decay=WD, max_grad_norm=1.0,
    fp16=USE_FP16, bf16=USE_BF16,
    # Epoch-based eval/save: one checkpoint per epoch, keeps 2.
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    greater_is_better=False, logging_steps=10, save_total_limit=2,
    report_to="none", seed=42,
    max_seq_length=MAX_SEQ_LENGTH, dataset_text_field="text",
    packing=True, padding_free=False, optim="adamw_8bit",
    dataloader_num_workers=0, dataloader_pin_memory=False,
)

trainer = SFTTrainer(model=model, tokenizer=tokenizer,
    train_dataset=train_dataset, eval_dataset=val_dataset, args=args)

# Resume from the latest checkpoint if one exists.
_resume = None
_ckpt_dir = Path(CKPT_DIR)
if _ckpt_dir.exists():
    _ckpts = sorted(
        [d for d in _ckpt_dir.iterdir()
         if d.is_dir() and d.name.startswith("checkpoint-")],
        key=lambda d: int(d.name.split("-")[-1]))
    if _ckpts:
        _resume = str(_ckpts[-1])
        print(f"Resuming from checkpoint: {_resume}")

print(f"Starting DG training | eff_batch={BATCH_SIZE*GRAD_ACCUM}")
start = time.time()
train_result = trainer.train(resume_from_checkpoint=_resume)
elapsed = round(time.time() - start, 1)
print(f"Done in {elapsed}s | loss={round(train_result.training_loss, 4)}")

## 9. Save LoRA Adapter


In [ ]:
ADAPTER_PATH = f"{OUTPUT_DIR}/adapter"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Saved to: {ADAPTER_PATH}")
for f in sorted(Path(ADAPTER_PATH).iterdir()):
    print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")


## 10. Evaluate (ROUGE-L + Cosine Similarity)


In [ ]:
import numpy as np
import gc
from rouge_score import rouge_scorer as rs_mod
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

FastLanguageModel.for_inference(model)
scorer = rs_mod.RougeScorer(["rougeL"], use_stemmer=True)
embedder = SentenceTransformer("all-MiniLM-L6-v2")
rL = []
mastery_cos = defaultdict(list)
ok = fail = 0
N = min(50, len(val_samples))
print(f"Evaluating on {N} samples...")

for s in val_samples[:N]:
    txt = s.get("text", "")

    # Extract correct answer and reference distractor from the formatted 'text'.
    ca_m  = re.search(r"answer: (.+?)(?:\|| content:)", txt)
    ref_m = re.search(r"DISTRACTOR:\s*(.+?)(?:\n|$)", txt)
    ca  = ca_m.group(1).strip()  if ca_m  else ""
    ref = ref_m.group(1).strip() if ref_m else ""

    # Extract question from formatted 'text'.
    q_m      = re.search(r"question: (.+?)(?:\| answer:)", txt)
    question = q_m.group(1).strip() if q_m else ""

    mastery   = s.get("mastery_level",        "unknown")
    qt        = s.get("question_type",        "4a")
    sc_cat    = s.get("score_category",       "moderate")
    dist_cat  = s.get("distractor_category",  "general")

    # Reconstruct chunk from 'text'.
    chunk = ""
    cm = re.search(r"\| content:\s*(.+?)(?=\n<\|im_end\|>|\Z)", txt, re.DOTALL)
    if cm:
        chunk = cm.group(1).strip()[:400]

    msgs = build_dg_eval_prompt(question, ca, qt, mastery, sc_cat, dist_cat, chunk)
    inp  = tokenizer(apply_chat(msgs, tokenizer), return_tensors="pt",
                     truncation=True, max_length=MAX_SEQ_LENGTH)
    inp  = {k: v.to(model.device) for k, v in inp.items()}
    ilen = inp["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=80, temperature=0.0, do_sample=False)

    raw = tokenizer.decode(out[0][ilen:], skip_special_tokens=True)
    p   = extract_dg_output(raw)
    if p:
        ok += 1
        if ref:
            rL.append(scorer.score(ref, p)["rougeL"].fmeasure)
        if ca:
            embs = embedder.encode([ca, p], convert_to_numpy=True, show_progress_bar=False)
            mastery_cos[mastery].append(
                float(cosine_similarity([embs[0]], [embs[1]])[0][0]))
    else:
        fail += 1
    del inp, out
    torch.cuda.empty_cache()

avg = lambda l: round(float(np.mean(l)), 4) if l else 0.0
print(f"\nROUGE-L={avg(rL)} | Parse={ok}/{N}")
print("Cosine sim per mastery:")
for m, v in sorted(mastery_cos.items()):
    print(f"  {m}: {avg(v)}")
gc.collect()
torch.cuda.empty_cache()

## 11. Save Metrics


In [ ]:
avg = lambda l: round(float(np.mean(l)), 4) if l else 0.0
metrics = {
    "task": "DG", "gpu": GPU_TYPE, "base_model": BASE_MODEL,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
    "train_loss": round(train_result.training_loss, 4),
    "training_time_s": elapsed,
    "rougeL": avg(rL), "parse_ok": ok, "parse_fail": fail,
    "train_n": len(train_samples), "val_n": len(val_samples),
    "warmup_steps": WARMUP, "total_steps": _total_steps,
}
mp = f"{OUTPUT_DIR}/training_metrics.json"
with open(mp, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics: {mp}")
print("Done! Download /kaggle/working/mcq_dg/ from the Output tab.")